# 12.7 CI/CD for ML

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 12: Model Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-12-model-deployment/12.07_ci-cd-for-ml.ipynb)

Companion code for Section 12.7 of the book. Run the cells in order from top to bottom.

**Example 12.7.1** · yaml  
Saved in this repo as `src/.github/workflows/ml_cicd.yml`.

```yaml
# .github/workflows/ml_cicd.yml
# Triggers: on push to main, pull request to main, scheduled weekly retraining

name: ML Pipeline CI/CD

on:
  push:
    branches: [main, develop]
  pull_request:
    branches: [main]
  schedule:
    - cron: '0 2 * * 1'  # Every Monday at 2 AM - scheduled retraining

env:
  PYTHON_VERSION: '3.11'
  MODEL_NAME: cancer-classifier
  MIN_ACCURACY: '0.95'
  MIN_AUC: '0.98'

jobs:
  # ─── Job 1: Code quality and tests ────────────────────────────
  test:
    name: Code Quality & Tests
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - name: Install dependencies
        run: |
          pip install -r requirements.txt
          pip install pytest pytest-cov flake8 black isort mypy

      - name: Lint (flake8)
        run: flake8 . --max-line-length=100 --exclude=.git,__pycache__

      - name: Format check (black)
        run: black --check .

      - name: Type check (mypy)
        run: mypy main.py --ignore-missing-imports

      - name: Unit tests
        run: pytest tests/ -v --cov=. --cov-report=xml

      - name: Upload coverage
        uses: codecov/codecov-action@v3

  # ─── Job 2: Model training and evaluation ──────────────────────
  train-evaluate:
    name: Train & Evaluate Model
    runs-on: ubuntu-latest
    needs: test
    outputs:
      model-accuracy: ${{ steps.evaluate.outputs.accuracy }}
      model-auc:      ${{ steps.evaluate.outputs.auc }}
      model-path:     ${{ steps.train.outputs.model_path }}

    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v4
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Train model
        id: train
        run: |
          python train.py --output-dir models/ --data-version ${{ github.sha }}
          echo "model_path=models/model_$(date +%Y%m%d_%H%M%S).joblib" >> $GITHUB_OUTPUT

      - name: Evaluate model
        id: evaluate
        run: |
          python evaluate.py --model-path ${{ steps.train.outputs.model_path }} \
            --output metrics.json
          # Extract metrics and set as outputs
          ACC=$(python -c "import json; d=json.load(open('metrics.json')); print(d['accuracy'])")
          AUC=$(python -c "import json; d=json.load(open('metrics.json')); print(d['auc'])")
          echo "accuracy=$ACC" >> $GITHUB_OUTPUT
          echo "auc=$AUC" >> $GITHUB_OUTPUT
          echo "Model accuracy: $ACC  AUC: $AUC"

      - name: Quality gate - fail if metrics below threshold
        run: |
          python -c "
          import json, sys
          metrics = json.load(open('metrics.json'))
          min_acc = float('${{ env.MIN_ACCURACY }}')
          min_auc = float('${{ env.MIN_AUC }}')
          if metrics['accuracy'] < min_acc:
              print(f'FAIL: accuracy {metrics[\"accuracy\"]:.4f} < {min_acc}'); sys.exit(1)
          if metrics['auc'] < min_auc:
              print(f'FAIL: AUC {metrics[\"auc\"]:.4f} < {min_auc}'); sys.exit(1)
          print(f'PASS: accuracy={metrics[\"accuracy\"]:.4f}  auc={metrics[\"auc\"]:.4f}')
          "

      - name: Upload model artifact
        uses: actions/upload-artifact@v3
        with:
          name: trained-model
          path: models/
          retention-days: 30

  # ─── Job 3: Build and push Docker image ───────────────────────
  build-docker:
    name: Build & Push Docker
    runs-on: ubuntu-latest
    needs: train-evaluate
    if: github.ref == 'refs/heads/main'

    steps:
      - uses: actions/checkout@v4

      - name: Download model artifact
        uses: actions/download-artifact@v3
        with:
          name: trained-model
          path: models/

      - name: Login to Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKERHUB_USERNAME }}
          password: ${{ secrets.DOCKERHUB_TOKEN }}

      - name: Build and push
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: |
            ${{ secrets.DOCKERHUB_USERNAME }}/${{ env.MODEL_NAME }}:latest
            ${{ secrets.DOCKERHUB_USERNAME }}/${{ env.MODEL_NAME }}:${{ github.sha }}
          cache-from: type=gha
          cache-to: type=gha,mode=max

  # ─── Job 4: Deploy to production ─────────────────────────────
  deploy:
    name: Deploy to Production
    runs-on: ubuntu-latest
    needs: build-docker
    if: github.ref == 'refs/heads/main'
    environment: production   # Requires manual approval in GitHub

    steps:
      - name: Deploy to Cloud Run
        run: |
          gcloud run deploy ${{ env.MODEL_NAME }} \
            --image ${{ secrets.DOCKERHUB_USERNAME }}/${{ env.MODEL_NAME }}:${{ github.sha }} \
            --platform managed \
            --region us-central1 \
            --memory 2Gi --cpu 2 \
            --min-instances 1 --max-instances 20

      - name: Smoke test deployment
        run: |
          ENDPOINT=$(gcloud run services describe ${{ env.MODEL_NAME }} \
            --region us-central1 --format='value(status.url)')
          python -c "
          import requests
          r = requests.get('$ENDPOINT/health')
          assert r.status_code == 200, f'Health check failed: {r.status_code}'
          assert r.json()['status'] == 'healthy', 'Model not healthy'
          print('Smoke test PASSED')"

      - name: Notify Slack on success
        uses: 8398a7/action-slack@v3
        with:
          status: success
          text: |
            Model ${{ env.MODEL_NAME }} deployed successfully!
            Accuracy: ${{ needs.train-evaluate.outputs.model-accuracy }}
            AUC: ${{ needs.train-evaluate.outputs.model-auc }}
        env:
          SLACK_WEBHOOK_URL: ${{ secrets.SLACK_WEBHOOK }}
        if: always()
```

**Example 12.7.2** · also saved as `src/train.py`

In [ ]:
# train.py - Reproducible training script for CI/CD
import argparse
import json
import os
import datetime
import joblib
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline

def train(output_dir: str, data_version: str, random_state: int = 42):
    print(f'Training model... data_version={data_version}')

    # Load data (replace with your data loading logic)
    data = load_breast_cancer()
    X, y = data.data, data.target

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=random_state
    )

    # Train
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(n_estimators=200, random_state=random_state))
    ])
    pipeline.fit(X_tr, y_tr)

    # Save
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    model_path = f'{output_dir}/model_{ts}.joblib'
    joblib.dump(pipeline, model_path)

    # Save metadata
    meta = {
        'model_path': model_path,
        'data_version': data_version,
        'created_at': datetime.datetime.now().isoformat(),
        'random_state': random_state,
        'n_train': len(X_tr),
        'n_test': len(X_te),
    }
    with open(f'{output_dir}/model_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    print(f'Model saved: {model_path}')
    return model_path


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--output-dir', default='models/')
    parser.add_argument('--data-version', default='local')
    parser.add_argument('--random-state', type=int, default=42)
    args = parser.parse_args()
    train(args.output_dir, args.data_version, args.random_state)

**Example 12.7.3** · writes `src/tests/test_api.py`

In [ ]:
%%writefile src/tests/test_api.py
# tests/test_api.py - Automated API tests for CI
import pytest
import numpy as np
from sklearn.datasets import load_breast_cancer
from fastapi.testclient import TestClient

# Import the app (adjust import path as needed)
from main import app

client = TestClient(app)
data   = load_breast_cancer()
VALID_FEATURES = data.data[0].tolist()
API_KEY = 'fastapi-secret-key'
HEADERS = {'X-API-Key': API_KEY}

class TestHealth:
    def test_health_returns_200(self):
        r = client.get('/health')
        assert r.status_code == 200

    def test_health_contains_status(self):
        r = client.get('/health')
        assert 'status' in r.json()
        assert r.json()['status'] in ['healthy', 'degraded']

class TestPredict:
    def test_valid_prediction(self):
        r = client.post('/predict',
                        json={'features': VALID_FEATURES},
                        headers=HEADERS)
        assert r.status_code == 200
        body = r.json()
        assert 'prediction' in body
        assert body['prediction'] in ['malignant', 'benign']
        assert 0 <= body['confidence'] <= 1
        assert body['latency_ms'] > 0

    def test_unauthorised_returns_401(self):
        r = client.post('/predict', json={'features': VALID_FEATURES})
        assert r.status_code == 401

    def test_wrong_feature_count_returns_400(self):
        r = client.post('/predict',
                        json={'features': [1.0, 2.0, 3.0]},
                        headers=HEADERS)
        assert r.status_code == 422  # Pydantic validation error

    def test_predictions_are_deterministic(self):
        r1 = client.post('/predict', json={'features': VALID_FEATURES}, headers=HEADERS)
        r2 = client.post('/predict', json={'features': VALID_FEATURES}, headers=HEADERS)
        assert r1.json()['prediction'] == r2.json()['prediction']
        assert r1.json()['confidence'] == r2.json()['confidence']

    def test_batch_prediction(self):
        samples = data.data[:10].tolist()
        r = client.post('/predict/batch',
                        json={'samples': samples},
                        headers=HEADERS)
        assert r.status_code == 200
        body = r.json()
        assert body['n_samples'] == 10
        assert len(body['predictions']) == 10

    def test_model_accuracy_meets_threshold(self):
        '''Integration test: model should exceed 95% accuracy on test set.'''
        samples   = data.data.tolist()
        r = client.post('/predict/batch', json={'samples': samples}, headers=HEADERS)
        predictions = r.json()['predictions']
        pred_labels = [0 if p=='malignant' else 1 for p in predictions]
        accuracy    = np.mean(np.array(pred_labels) == data.target)
        assert accuracy >= 0.95, f'Accuracy {accuracy:.4f} below 0.95 threshold'

    def test_latency_under_100ms(self):
        r = client.post('/predict', json={'features': VALID_FEATURES}, headers=HEADERS)
        assert r.json()['latency_ms'] < 100, 'Prediction too slow'

Run this from a terminal (not inside the notebook):

```bash
pytest src/tests/test_api.py
```